In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from cmdstanpy import CmdStanModel
import cmdstanpy
import numpy as np
from scipy.stats import invgamma
from scipy.stats import norm
from scipy.special import softmax
from gptools.stan import get_include

import pandas as pd
from glob import glob
from nteprsm import utils 
from cmdstanpy import stanfit
from settings import ROOT_DIR
import plotly.express as px
import utils as notebook_utils
# use customize plotly template
notebook_utils.set_custom_template()

import pickle


In [ ]:
filepath = ROOT_DIR/"QUALITY_NJ2"
df = pd.read_csv('data/raw/quality_nj2.csv')  # Replace 'file.csv' with your file path
df.columns = [col.lower() for col in df.columns]
df = df.assign(
    entry_name_code=pd.Categorical(df["entry_name"]).codes,
    plt_id_code=pd.Categorical(df["plt_id"]).codes,
    rater_code=pd.Categorical(df["rater"]).codes,
    rating_event_code=pd.Categorical(df["rating_event"]).codes,
)
df["entry_cumcount"] = df.groupby("entry_name").cumcount() + 1

In [ ]:
# load model configuration
config_file = ROOT_DIR/"config/nteprsm_njkbg07.yml"
config = utils.load_config(config_file)
config["sampling"]['save_warmup'] = False
# process data
datahandler = utils.DataHandler(filepath='data/raw/quality_nj2.csv')
datahandler.model_data = df
datahandler.load_data()
datahandler.preprocess_data()
datahandler.generate_stan_data(**config["stan_additional_data"])

In [ ]:
nteprsm = CmdStanModel(
    stan_file=config["stan_file"],
    stanc_options={"include-paths": get_include()},
)
fit = nteprsm.sample(data=datahandler.stan_data, **config["sampling"])

In [ ]:
# Save to a file
with open('annual_seasonality_nj2.pkl', 'wb') as file:
    pickle.dump(fit, file)